# 05b - Topic Assignment via Zero-Shot Categories

Instead of letting BERTopic find topics purely data-driven (52 fine-grained clusters, see `05_topic_analysis.ipynb`), this notebook assigns each speech to one of a small set of **predefined, coarse policy categories** (roughly matching Bundestag committee areas). Every speech is assigned to its nearest category by cosine similarity — there is no `-1`/unassigned bucket, since every speech gets a category.

The category embeddings (`zeroshot_topic_embeddings.parquet`) were produced with the same fine-tuned model used for the speech embeddings, so both live in the same vector space.

In [1]:
import unicodedata
import numpy as np
import pandas as pd
from sklearn.metrics.pairwise import cosine_similarity

## 1. Load the data

Speech embeddings and the pre-computed category embeddings.

In [ ]:
# full fine-tuned embedding dataset, already containing the speech texts
df = pd.read_parquet("jina_v3_contrastive_full.parquet")  
df = df[df["speechContent"].notna() & (df["speechContent"].str.strip() != "")].reset_index(drop=True)

embeddings = np.array(df["embedding"].tolist(), dtype="float32")

print(f"{len(df):,} speeches")
print("columns:", list(df.columns))

82,447 speeches
columns: ['id', 'speechContent', 'politicianId', 'politicianName', 'firstName', 'lastName', 'positionShort', 'factionId', 'party', 'date', 'year', 'year_month', 'n_tokens', 'used_in_finetune', 'used_in_party_probe_val', 'used_in_party_probe_test', 'used_finetuned_weights', 'embedding']


In [3]:
# category embeddings, produced with the same (fine-tuned) model
df_categories = pd.read_parquet("zeroshot_topic_embeddings.parquet")

zeroshot_topic_list = df_categories["topic"].tolist()
category_embeddings = np.stack(df_categories["embedding"])

print(f"{len(zeroshot_topic_list)} categories")
df_categories

21 categories


,topic,embedding
0,Außenpolitik und internationale Beziehungen,"[0.09617888, -0.15636098, 0.065269716, -0.0332..."
1,Innere Sicherheit und Polizei,"[0.018960787, -0.15969495, 0.0421052, 0.036920..."
2,"Migration, Asyl und Einwanderung","[0.041229006, -0.17929918, 0.13565937, 0.01634..."
3,Sport und Ehrenamt,"[0.012529727, -0.17042173, 0.05671904, 0.01456..."
4,Recht und Verbraucherschutz,"[0.042030547, -0.06627503, 0.100224085, 0.0093..."
5,Steuern und Finanzpolitik,"[0.09679413, -0.04804805, 0.016609225, 0.04112..."
6,Bundeshaushalt und öffentliche Ausgaben,"[0.007676577, -0.050209872, 0.06350469, 0.0555..."
7,Wirtschaftspolitik und Energieversorgung,"[0.06390001, -0.05812556, -0.04786639, -0.0437..."
8,Landwirtschaft und Ernährung,"[0.050605208, -0.1195316, 0.13865988, -0.04394..."
9,Arbeitsmarkt und Sozialpolitik,"[0.032052148, -0.13831486, 0.015475224, -0.001..."


## 2. Assign each speech to its nearest category

Cosine similarity between every speech and every category embedding, then the argmax per speech — every speech gets assigned, no leftover `-1` bucket.

In [4]:
speech_embeddings_norm = embeddings / np.linalg.norm(embeddings, axis=1, keepdims=True)
category_embeddings_norm = category_embeddings / np.linalg.norm(category_embeddings, axis=1, keepdims=True)

similarities = cosine_similarity(speech_embeddings_norm, category_embeddings_norm)
assigned_idx = similarities.argmax(axis=1)

df["topic_emb_raw"] = assigned_idx
df["topic_emb_full"] = assigned_idx  # no unassigned bucket here, so raw == full

print(pd.Series(assigned_idx).map(dict(enumerate(zeroshot_topic_list))).value_counts())

Umweltschutz, Klimapolitik und Atomkraft       9750
Bildung, Familie und Jugend                    9160
Arbeitsmarkt und Sozialpolitik                 8166
Steuern und Finanzpolitik                      6935
Europapolitik und EU-Angelegenheiten           6589
Verteidigungspolitik und Bundeswehr            5547
Gesundheitspolitik und Krankenversicherung     4841
Außenpolitik und internationale Beziehungen    4369
Menschenrechte und humanitäre Hilfe            3845
Innere Sicherheit und Polizei                  3462
Landwirtschaft und Ernährung                   3233
Recht und Verbraucherschutz                    2778
Migration, Asyl und Einwanderung               2552
Wirtschaftspolitik und Energieversorgung       2484
Verkehr und Infrastruktur                      2281
Wohnungspolitik und Städtebau                  1854
Forschung, Technologie und Raumfahrt           1845
Kultur und Medien                              1337
Entwicklungszusammenarbeit                      648
Sport und Eh

## 3. Save the assignments

Same output format as `05_topic_analysis.ipynb`'s `speech_topic_assignments.csv`, so the trajectory notebook can read either file without changes.

In [5]:
topic_names = dict(enumerate(zeroshot_topic_list))

out = (df[["politicianId", "party", "id", "topic_emb_raw", "topic_emb_full", "date"]]
       .rename(columns={"politicianId": "speaker_id", "id": "speech_id"}))
out["topic_emb_raw_name"] = out["topic_emb_raw"].map(topic_names)
out["topic_emb_full_name"] = out["topic_emb_full"].map(topic_names)
out = out[["speaker_id", "party", "speech_id",
           "topic_emb_raw", "topic_emb_raw_name",
           "topic_emb_full", "topic_emb_full_name", "date"]]

out.to_csv("speech_topic_assignments_zeroshot.csv", index=False, encoding="utf-8")
print(f"wrote {len(out):,} rows to speech_topic_assignments_zeroshot.csv")
out.head()

wrote 82,447 rows to speech_topic_assignments_zeroshot.csv


,speaker_id,party,speech_id,topic_emb_raw,topic_emb_raw_name,topic_emb_full,topic_emb_full_name,date
0,11001434,SPD,604683,14,"Umweltschutz, Klimapolitik und Atomkraft",14,"Umweltschutz, Klimapolitik und Atomkraft",2000-01-19
1,11002754,CDU/CSU,604883,11,"Bildung, Familie und Jugend",11,"Bildung, Familie und Jugend",2000-01-20
2,11000431,SPD,604885,11,"Bildung, Familie und Jugend",11,"Bildung, Familie und Jugend",2000-01-20
3,11001180,FDP,604905,10,Verteidigungspolitik und Bundeswehr,10,Verteidigungspolitik und Bundeswehr,2000-01-20
4,11003216,CDU/CSU,605067,10,Verteidigungspolitik und Bundeswehr,10,Verteidigungspolitik und Bundeswehr,2000-01-21
